# Arquitectura U-Net

In [ ]:
# Importar las bibliotecas necesarias
from pathlib import Path
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib import pyplot as plt

In [ ]:
# Abrir imagen de prueba
ruta_imagen = Path("../data/processed/images/Sphynx/Sphynx_176.jpg")
imagen = Image.open(ruta_imagen)
imagen_gris = imagen.convert("L")  # Convertir a escala de grises
# Convertir a tensor, cambiar el orden de los canales y agregar dimensión de batch
imagen_tensor = torch.from_numpy(np.array(imagen_gris)).unsqueeze(0).unsqueeze(0).float() / 255.0
print("\nImagen de prueba cargada y convertida a tensor:")
print("Dimensiones de la imagen de prueba:", imagen_tensor.shape)

## 1. Ejemplo de implementación de capas convolucionales

In [ ]:
# Definir capas convolucionales
difuminacion = nn.Conv2d(
	in_channels=1,
	out_channels=1,
	kernel_size=3,
	padding=1,
	bias=False
)

bordes = nn.Conv2d(
	in_channels=1,
	out_channels=1,
	kernel_size=3,
	padding=1,
	bias=False
)

In [ ]:
# Definir kernels 
kernel_promedio = torch.ones(3, 3) / 9

kernel_sobel = torch.tensor([
	[-1., 0., 1.],
	[-2., 0., 2.],
	[-1., 0., 1.]
])

with torch.no_grad():
	difuminacion.weight.copy_(
		kernel_promedio.reshape(1, 1, 3, 3)
	)
	bordes.weight.copy_(
		kernel_sobel.reshape(1, 1, 3, 3)
	)

In [ ]:
#Agrupar operaciones mediante Sequential
procesamiento = nn.Sequential(
	difuminacion,
	bordes,
	nn.ReLU()
)

# Desactivar gradientes: solo queremos aplicar filtros
for parametro in procesamiento.parameters():
	parametro.requires_grad_(False)

In [ ]:
# Ejecutar el procesamiento
with torch.no_grad(): # desactivar el cálculo de gradientes
	resultado = procesamiento(imagen_tensor)

print("Entrada:", imagen_tensor.shape)
print("Salida:", resultado.shape)

print("Pesos totales:",
	  sum(p.numel() for p in procesamiento.parameters()))

print("Parámetros entrenables:",
	  sum(p.numel() for p in procesamiento.parameters()
		  if p.requires_grad))

In [ ]:
# Guardar las salidas intermedias
x = imagen_tensor

salidas = [x]
nombres = [
	"Original",
	"Difuminación",
	"Sobel",
	"ReLU"
]

with torch.no_grad():
	for capa in procesamiento:
		x = capa(x)
		salidas.append(x)

# Visualizar cada etapa
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, (ax, tensor) in enumerate(zip(axes, salidas)):
	imagen_mostrar = tensor[0, 0].numpy()

	if i == 2:
		# Escala simétrica para distinguir bordes positivos y negativos
		limite = abs(imagen_mostrar).max()
		ax.imshow(
			imagen_mostrar,
			cmap="gray",
			vmin=-limite,
			vmax=limite
		)
	else:
		ax.imshow(imagen_mostrar, cmap="gray")

	ax.set_title(nombres[i])
	ax.axis("off")

plt.tight_layout()
plt.show()


## 2. Arquitectura mini U-Net

### Bloque de doble convolución para el encoder

In [ ]:
# Definir un bloque de convolución doble
class DoubleConv(nn.Module):
	def __init__(self, in_channels, out_channels):
		super(DoubleConv, self).__init__()

		self.bloque = nn.Sequential(
			nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
			nn.ReLU(inplace=True),
			nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
			nn.ReLU(inplace=True)
		)

	def forward(self, x):
		return self.bloque(x)

### Arquitectura mini U-Net

In [ ]:
class MiniUNet(nn.Module):
	def __init__(self, in_channels=3, num_classes=1):
		super().__init__()

		# Encoder
		self.enc1 = DoubleConv(in_channels, 16)
		self.enc2 = DoubleConv(16, 32)

		self.pool = nn.MaxPool2d(2)

		# Bottleneck
		self.bottleneck = DoubleConv(32, 64)

		# Decoder
		self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
		self.dec2 = DoubleConv(64, 32)

		self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
		self.dec1 = DoubleConv(32, 16)

		# Capa de salida
		self.final = nn.Conv2d(16, num_classes, 1)

	def forward(self, x):

		print("Entrada:       ", tuple(x.shape))
		
		# Encoder: guardar características para skip connections
		e1 = self.enc1(x)
		print("Encoder 1:     ", tuple(e1.shape))
		e2 = self.enc2(self.pool(e1))
		print("Encoder 2:     ", tuple(e2.shape))
		# Bottleneck
		b = self.bottleneck(self.pool(e2))
		print("Bottleneck:    ", tuple(b.shape))

		# Decoder: upsampling + concatenación
		d2 = self.up2(b)
		print("Upsampling 2:  ", tuple(d2.shape))
		d2 = torch.cat((e2, d2), dim=1)
		print("Concat 2:      ", tuple(d2.shape))
		d2 = self.dec2(d2)
		print("Decoder 2:     ", tuple(d2.shape))

		d1 = self.up1(d2)
		print("Upsampling 1:  ", tuple(d1.shape))
		d1 = torch.cat((e1, d1), dim=1)
		print("Concat 1:      ", tuple(d1.shape))
		d1 = self.dec1(d1)
		print("Decoder 1:     ", tuple(d1.shape))

		return self.final(d1)


In [ ]:
mini_unet = MiniUNet(in_channels=1, num_classes=1)
x = torch.randn(1, 1, 256, 256)

mini_unet.eval()

with torch.no_grad():
    y = mini_unet(x)

## 3. Arquitectura U-Net completa

In [ ]:
# Definir la arquitectura completa de U-Net
class UNet(nn.Module):
	def __init__(self, in_channels=3, num_classes=1):
		super().__init__()

		# Encoder
		self.enc1 = DoubleConv(in_channels, 64)
		self.enc2 = DoubleConv(64, 128)
		self.enc3 = DoubleConv(128, 256)
		self.enc4 = DoubleConv(256, 512)

		self.pool = nn.MaxPool2d(2)

		# Bottleneck
		self.bottleneck = DoubleConv(512, 1024)

		# Decoder
		self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
		self.dec4 = DoubleConv(1024, 512)

		self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
		self.dec3 = DoubleConv(512, 256)

		self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
		self.dec2 = DoubleConv(256, 128)

		self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
		self.dec1 = DoubleConv(128, 64)

		# Capa de salida
		self.final = nn.Conv2d(64, num_classes, 1)

	def forward(self, x):
		# Encoder
		e1 = self.enc1(x)
		print("Encoder 1:     ", tuple(e1.shape))
		e2 = self.enc2(self.pool(e1))
		print("Encoder 2:     ", tuple(e2.shape))
		e3 = self.enc3(self.pool(e2))
		print("Encoder 3:     ", tuple(e3.shape))
		e4 = self.enc4(self.pool(e3))
		print("Encoder 4:     ", tuple(e4.shape))

		# Bottleneck
		b = self.bottleneck(self.pool(e4))
		print("Bottleneck:    ", tuple(b.shape))

		# Decoder
		d4 = self.up4(b)
		print("Upsampling 4:  ", tuple(d4.shape))
		d4 = torch.cat((e4, d4), dim=1)
		print("Concat 4:      ", tuple(d4.shape))
		d4 = self.dec4(d4)
		print("Decoder 4:     ", tuple(d4.shape))

		d3 = self.up3(d4)
		print("Upsampling 3:  ", tuple(d3.shape))
		d3 = torch.cat((e3, d3), dim=1)
		print("Concat 3:      ", tuple(d3.shape))
		d3 = self.dec3(d3)
		print("Decoder 3:     ", tuple(d3.shape))

		d2 = self.up2(d3)
		print("Upsampling 2:  ", tuple(d2.shape))
		d2 = torch.cat((e2, d2), dim=1)
		print("Concat 2:      ", tuple(d2.shape))
		d2 = self.dec2(d2)
		print("Decoder 2:     ", tuple(d2.shape))

		d1 = self.up1(d2)
		print("Upsampling 1:  ", tuple(d1.shape))
		d1 = torch.cat((e1, d1), dim=1)
		print("Concat 1:      ", tuple(d1.shape))
		d1 = self.dec1(d1)
		print("Decoder 1:     ", tuple(d1.shape))

		return self.final(d1)


In [ ]:
unet = UNet(in_channels=1, num_classes=1)
x = torch.randn(1, 1, 256, 256)
unet.eval()
with torch.no_grad():
	y = unet(x)